# VocabOps — GPU POS Tagging & Collocation Extraction (Colab Runner)

Thin reproducible runner for `python -m vocabops.enrich` (see [Issue #23](https://github.com/hamakyo/vocabops/issues/23)).

**Guardrails**
- All NLP and aggregation logic lives in `src/vocabops/enrich.py`; this notebook only installs the repo and runs CLI commands.
- Store raw third-party corpora and intermediate files in Google Drive or `/content/work` — never commit raw text or `out/` to GitHub.
- Use a fixed `SNAPSHOT_ID` for deterministic, byte-identical outputs.

In [ ]:
#@title 1. Run parameters
REPO_URL = "https://github.com/hamakyo/vocabops.git" #@param {type:"string"}
GIT_REF = "main" #@param {type:"string"}
USE_GOOGLE_DRIVE = False #@param {type:"boolean"}
DRIVE_WORKDIR = "/content/drive/MyDrive/vocabops" #@param {type:"string"}
MODEL_NAME = "en_core_web_trf" #@param ["en_core_web_trf", "en_core_web_sm"]
DEVICE = "auto" #@param ["auto", "gpu", "cpu"]
SNAPSHOT_ID = "human-se-v0.1-proto-enrich" #@param {type:"string"}
INPUT_PATH = "data/fixtures/sample_corpus" #@param {type:"string"}
TERMS_PATH = "" #@param {type:"string"}
MIN_DF = 1 #@param {type:"integer"}
MIN_COLLOCATION_COUNT = 1 #@param {type:"integer"}
BATCH_SIZE = 64 #@param {type:"integer"}

import subprocess

gpu_check = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu_check.returncode == 0:
    print("GPU detected:", gpu_check.stdout.strip())
else:
    print("No GPU detected; will fall back to CPU if DEVICE='auto'.")


In [ ]:
#@title 2. Mount Drive (optional), clone repo, and install .[colab]
import os
import subprocess
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_ROOT = Path(DRIVE_WORKDIR)
else:
    WORK_ROOT = Path("/content/work")

WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path("/content/vocabops")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--prune"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", GIT_REF], check=False)

os.chdir(REPO_DIR)
subprocess.run(["pip", "install", "-q", "-e", ".[colab]"], check=True)
subprocess.run(["python", "-m", "spacy", "download", MODEL_NAME], check=True)

print("Checked out:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("Work root:  ", WORK_ROOT)


In [ ]:
#@title 3. Run python -m vocabops.enrich
import subprocess
from pathlib import Path

OUTPUT_DIR = WORK_ROOT / "out" / SNAPSHOT_ID
cmd = [
    "python",
    "-m",
    "vocabops.enrich",
    "--input",
    INPUT_PATH,
    "--output",
    str(OUTPUT_DIR),
    "--snapshot-id",
    SNAPSHOT_ID,
    "--model",
    MODEL_NAME,
    "--device",
    DEVICE,
    "--batch-size",
    str(BATCH_SIZE),
    "--min-df",
    str(MIN_DF),
    "--min-collocation-count",
    str(MIN_COLLOCATION_COUNT),
]
if TERMS_PATH.strip():
    cmd.extend(["--terms", TERMS_PATH.strip()])

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
#@title 4. Inspect manifest.json and top enrichment rows
import json

manifest_path = OUTPUT_DIR / "manifest.json"
enrichment_path = OUTPUT_DIR / "enrichment.jsonl"

print("=== manifest.json ===")
print(manifest_path.read_text(encoding="utf-8"))

print("=== top 20 enrichment rows ===")
with enrichment_path.open("r", encoding="utf-8") as handle:
    for idx, line in enumerate(handle):
        if idx >= 20:
            break
        row = json.loads(line)
        collocs = [f"{c['phrase']} ({c['pattern']}:{c['count']})" for c in row["collocations"][:4]]
        print(
            f"{row['id']:<24} df={row['df']:<4} tf={row['tf']:<5} "
            f"share={row['pos_share']:.2f}  forms={row['forms'][:4]}  collocs={collocs}"
        )
